# Refracción atmosférica: desarrollo completo de la actividad

**Curso:** Introducción a la Astronomía Práctica  
**Tema:** Efecto de la atmósfera en la posición y forma aparente de objetos celestes

En este cuaderno se desarrolla de forma extensa la actividad de la carpeta **RefraccionAtmosferica**.  
Se incluyen datos realistas, análisis cuantitativo, gráficas e interpretación física.


## 1. Contexto físico

La atmósfera terrestre no tiene densidad uniforme: al acercarnos al suelo, la presión y la densidad del aire son mayores. Por esta razón, la luz de un astro no viaja en línea recta dentro de la atmósfera, sino en una trayectoria curva. El resultado observable es que:

- un objeto cerca del horizonte **se ve más alto** de lo que realmente está;
- la parte inferior de discos extendidos (Sol/Luna) sufre más refracción que la parte superior, produciendo **achatamiento vertical**;
- el amanecer se adelanta y el atardecer se retrasa algunos minutos.

En astronomía de posición, estas diferencias son suficientemente grandes para producir errores relevantes si no se corrigen.


## 2. Objetivos

1. Cuantificar el achatamiento del disco solar/lunar en función de la altura angular.
2. Comparar alturas aparentes vs. geométricas para estimar el corrimiento por refracción.
3. Contrastar mediciones con un modelo matemático simple.
4. Discutir implicaciones prácticas para observación y astrometría.


## 3. Ecuación de trabajo

Usaremos la aproximación indicada en la guía:

\[
R[\text{arcmin}] = \cot\left(h + \frac{7.31}{h + 4.4}\right)\left(\frac{P}{101\,\text{kPa}}\right)\left(\frac{273}{273 + T}\right)
\]

donde:

- \(h\): altura aparente en grados,
- \(P\): presión atmosférica en kPa,
- \(T\): temperatura en \(^\circ\)C.

Para convertir a grados: \(R_{deg}=R/60\).


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    plt.style.use('seaborn-v0_8-whitegrid')
except Exception:
    plt.style.use('classic')


def refraccion_arcmin(h_deg, P_kpa=84.0, T_c=18.0):
    """Aproximación de refracción atmosférica (arcmin)."""
    h = np.asarray(h_deg, dtype=float)
    arg_deg = h + 7.31/(h + 4.4)
    R = 1/np.tan(np.deg2rad(arg_deg))
    R *= (P_kpa/101.0) * (273.0/(273.0 + T_c))
    return R


## 4. Experimento 0: deformación del disco (datos realistas)

Se simula una secuencia fotográfica de puesta del Sol con filtro solar. Los diámetros en píxeles son coherentes con una misma distancia focal, donde el diámetro horizontal cambia poco y el vertical aumenta con la altura por disminución de la refracción diferencial.

**Condiciones de observación supuestas (realistas):**

- Ciudad: Medellín (\(\sim 1500\,m\) s.n.m.)
- Temperatura ambiente: \(18\,^\circ\text{C}\)
- Presión local: \(84\,\text{kPa}\)
- Intervalo temporal entre imágenes: 4 a 6 minutos


In [ ]:
# Tabla 0: mediciones del disco en pixeles
altura_deg = np.array([1.2, 2.5, 4.0, 5.8, 7.6, 9.5, 12.0, 15.0, 18.5])
ancho_px   = np.array([1888, 1887, 1889, 1886, 1888, 1887, 1888, 1887, 1888])
alto_px    = np.array([1548, 1623, 1698, 1750, 1789, 1814, 1846, 1868, 1881])

ratio = ancho_px / alto_px

tabla0 = pd.DataFrame({
    'Hora local': ['17:58','18:03','18:08','18:13','18:18','18:24','18:30','18:37','18:45'],
    'Altura h (deg)': altura_deg,
    'Ancho W (px)': ancho_px,
    'Alto hv (px)': alto_px,
    'W/hv': np.round(ratio, 3)
})

tabla0


In [ ]:
fig, ax = plt.subplots(figsize=(8,4.8))
ax.plot(altura_deg, ratio, 'o-', color='tab:orange', lw=2)
ax.axhline(1.0, color='k', ls='--', lw=1)
ax.set_xlabel('Altura sobre el horizonte (deg)')
ax.set_ylabel('Relación de aspecto W/hv')
ax.set_title('Achatamiento aparente del disco vs altura')
plt.show()


### Interpretación del Experimento 0

- Cerca del horizonte (\(h\approx 1\) a \(3^\circ\)) la razón \(W/h_v\) es claramente mayor que 1, lo que confirma compresión vertical.
- Al subir de altura, \(W/h_v\rightarrow 1\), indicando que el disco recupera forma casi circular.
- En esta serie, la deformación se vuelve poco perceptible visualmente para \(h\gtrsim 12^\circ\), donde la razón ya es cercana a 1.02 o menor.


## 5. Experimento 1: alturas aparentes vs reales

Se toman tres estrellas brillantes visibles a baja altura. Se registra su altura geométrica (sin atmósfera) y su altura aparente (con atmósfera), como se haría en Stellarium alternando la opción de atmósfera.

Para consistencia con Medellín, se usa \(P=84\,\text{kPa}\), \(T=18^\circ\text{C}\) al calcular la ecuación.


In [ ]:
# Tabla 1: estrellas a baja altura
estrellas = ['Fomalhaut', 'Sirius', 'Aldebarán']
a_real = np.array([5.0, 15.0, 25.0])  # altura geométrica (sin atmósfera)

R_arcmin = refraccion_arcmin(a_real, P_kpa=84.0, T_c=18.0)
R_deg = R_arcmin / 60.0
a_ap = a_real + R_deg

tabla1 = pd.DataFrame({
    'Estrella': estrellas,
    'Altura real (deg)': np.round(a_real, 3),
    'Altura aparente (deg)': np.round(a_ap, 3),
    'Delta a (deg)': np.round(a_ap - a_real, 3),
    'Refracción R (arcmin)': np.round(R_arcmin, 2)
})

tabla1


In [ ]:
fig, ax = plt.subplots(figsize=(8,4.8))
ax.plot(a_real, R_arcmin, 's-', color='tab:blue', lw=2)
ax.set_xlabel('Altura real (deg)')
ax.set_ylabel('Refracción (arcmin)')
ax.set_title('Disminución de la refracción con la altura')
plt.show()


### Interpretación del Experimento 1

- La refracción calculada es mayor a baja altura y cae rápidamente al aumentar \(h\).
- Una estrella a \(5^\circ\) puede desplazarse varios minutos de arco, mientras que a \(25^\circ\) el corrimiento es bastante menor.
- Este comportamiento coincide con la física del problema: los rayos cercanos al horizonte atraviesan más atmósfera efectiva.


## 6. Comparación Barranquilla vs Medellín (pregunta 2)

Para una misma estrella baja en el horizonte, comparamos el modelo en dos condiciones realistas:

- **Barranquilla (casi nivel del mar):** \(P\approx 101\,\text{kPa},\; T\approx 30^\circ C\)
- **Medellín (montaña):** \(P\approx 84\,\text{kPa},\; T\approx 18^\circ C\)


In [ ]:
h_test = np.array([5.0, 10.0, 15.0])
R_baq = refraccion_arcmin(h_test, P_kpa=101.0, T_c=30.0)
R_med = refraccion_arcmin(h_test, P_kpa=84.0, T_c=18.0)

comparacion = pd.DataFrame({
    'h (deg)': h_test,
    'R Barranquilla (arcmin)': np.round(R_baq, 2),
    'R Medellín (arcmin)': np.round(R_med, 2),
    'Diferencia BAQ-MDE (arcmin)': np.round(R_baq - R_med, 2)
})

comparacion


Se observa que, para la misma altura, Barranquilla presenta refracción algo mayor por su presión más alta. Por tanto, **no medirían exactamente la misma altura aparente**.


## 7. Respuestas a las preguntas de análisis y conclusiones

### 7.1 Tendencia de \(W/h_v\) vs altura
La tendencia es decreciente hacia 1. Esto indica que la deformación vertical es máxima junto al horizonte y disminuye con la altura. En esta serie, alrededor de \(12^\circ\) a \(15^\circ\) el achatamiento ya es visualmente pequeño.

### 7.2 ¿Misma altura aparente en Barranquilla y Medellín?
No. De acuerdo con la ecuación, \(R\) escala aproximadamente con \(P\) y también depende de \(T\). Como Barranquilla tiene mayor presión media, la refracción es mayor y la estrella se observará ligeramente más alta respecto a Medellín (si se ignoran diferencias de geometría por latitud/longitud).

### 7.3 Implicaciones para medición de asteroides
No es despreciable. Un error de pocos minutos de arco en posición angular puede trasladarse a incertidumbres grandes al estimar órbitas, especialmente en observaciones de seguimiento para objetos cercanos a la Tierra. La calidad del ajuste orbital depende de residuales angulares pequeños; por eso la corrección de refracción es parte estándar de la reducción astrométrica.

### 7.4 Refracción no monocromática (dispersión atmosférica)
La atmósfera es dispersiva: la luz azul (menor longitud de onda) se refracta más que la roja. Cerca del horizonte, una estrella puede mostrar un borde azulado hacia la parte más elevada y rojizo hacia la parte inferior, produciendo separación cromática vertical. Esto también afecta medidas fotométricas y astrométricas si no se usan correcciones o instrumentos adecuados.

---

## Conclusión general
La refracción atmosférica modifica tanto la **posición** como la **forma aparente** de los objetos celestes. Los datos y gráficas presentados confirman dos hechos clave: (i) la refracción crece fuertemente al aproximarse al horizonte, y (ii) la refracción diferencial deforma discos extensos como Sol/Luna. En práctica astronómica, corregir este efecto no es opcional: es indispensable para lograr mediciones confiables.
